# 04 · ¿Cuánto dependen los resultados de los supuestos?

Partimos de las tres reglas sin filtrar. Ahora comprobamos qué cambia al modificar costes o dividendos, manteniendo las mismas señales y meses de 1927–1989. No buscamos el supuesto que dé más beneficio.

Usamos Shiller y RF ya preparados, igual que en el notebook 03. No es necesario ejecutarlo antes, pero son necesarias las entradas de los notebooks 01 y 02. La evaluación desde 1990 sigue deshabilitada. Los resultados conservan la limitación de precios medios y dividendos reconstruidos.


## 1. Ver qué vamos a comparar

El central procede de los supuestos centrales en los supuestos visibles del cuaderno. Cada alternativa cambia **un supuesto** y deja los demás como en el central:

- Negociación: 0 %, 0,10 %, 0,25 % y 0,50 % del nominal negociado.
- Préstamo: 0,25 %, 1 % y 3 % anual, solo mientras haya posición corta.
- Dividendos aproximados, incluidos o excluidos.

Con los supuestos centrales son siete escenarios, porque el central no se repite. Si cambias la tarifa central en este cuaderno fuera de esas listas, se conserva también. No se prueban todas las combinaciones. Son supuestos hipotéticos ya declarados, no tarifas históricas reconstruidas.


In [ ]:
from tfm.data import project_root, save_result
from tfm.sensitivity import run_sensitivities
from tfm.simulation import WARNINGS as SIMULATION_WARNINGS
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from tfm.data import read_result
from tfm.sensitivity import sensitivity_scenarios
from tfm.simulation import STRATEGIES

ROOT = project_root()
parametros = {"initial_capital": 100.0, "transaction_cost_rate": 0.001,
              "annual_borrow_rate": 0.01, "include_dividends": True}
componentes, _ = read_result(ROOT, "monthly_return_components")
senales, _ = read_result(ROOT, "shiller_signals")
escenarios = sensitivity_scenarios(parametros)
display(pd.DataFrame([
    {"Escenario": caso["scenario_label"],
     "Negociación (%)": 100 * caso["parameters"]["transaction_cost_rate"],
     "Préstamo anual (%)": 100 * caso["parameters"]["annual_borrow_rate"],
     "Dividendos aprox.": caso["parameters"]["include_dividends"]}
    for caso in escenarios
]))
print(f"Mismos meses en todos los casos: {componentes.period.iloc[0]} → {componentes.period.iloc[-1]}")
print(f"Capital inicial por cuenta: {parametros['initial_capital']:g} unidades")


## 2. Ejecutar y comparar

**Esta celda guarda o sustituye** `data/processed/baseline_sensitivity.parquet` y su JSON. No modifica fuentes ni reglas y conserva `baseline_simulations`. Reutiliza el mismo simulador de los notebooks, incluida la liquidación final y el reajuste de los cortos.

La tabla guarda una fila por escenario y estrategia, no las cuentas mensuales adicionales. Para entender la contabilidad de un mes, utiliza el notebook 03. Si algún escenario resulta insolvente, el proceso se detiene sin omitirlo ni publicar una comparación parcial.

La columna «Cambio (p.p.)» es la diferencia de rentabilidad anualizada respecto al central. Pasar de 5 % a 4 % significa −1 punto porcentual, no −1 % de cambio relativo. Las caídas desde máximos se muestran como valores negativos.


In [ ]:
comparacion = run_sensitivities(componentes, senales, parametros)
informe = {"status": "passed", "rows": len(comparacion), "months_per_strategy": len(componentes),
          "first_period": componentes.period.iloc[0], "last_period": componentes.period.iloc[-1],
          "parameters": parametros,
          "scenarios": sensitivity_scenarios(parametros),
          "comparisons": comparacion.astype(object).where(pd.notna(comparacion), None).to_dict("records"),
          "warnings": [*SIMULATION_WARNINGS,
                       "Un supuesto cada vez; no es una búsqueda de parámetros ni certifica robustez fuera de muestra."]}
resumen = save_result(ROOT, "baseline_sensitivity", comparacion, informe)
print(f"{len(informe['scenarios'])} escenarios × {len(STRATEGIES)} estrategias = {len(comparacion)} resúmenes")
print(resumen["path"])
for estrategia, nombre in STRATEGIES.items():
    filas = comparacion.loc[comparacion.strategy.eq(estrategia)]
    vista = filas[["scenario_label", "annualized_return", "annualized_return_delta", "max_drawdown", "sharpe"]].copy()
    for columna in ["annualized_return", "annualized_return_delta", "max_drawdown"]:
        vista[columna] *= 100
    vista = vista.rename(columns={"scenario_label": "Escenario", "annualized_return": "Anualizada (%)",
                                 "annualized_return_delta": "Cambio (p.p.)", "max_drawdown": "Caída máxima (%)",
                                 "sharpe": "Sharpe"}).set_index("Escenario")
    print(nombre)
    display(vista.round(4))


## 3. Ver los cambios de rentabilidad anualizada

Cada barra mantiene el orden de escenarios declarado. La línea discontinua indica el resultado central de esa estrategia. Los ejes horizontales pueden tener escalas distintas; las cifras y sus unidades permiten comparar.


In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(13, 5), sharey=True, layout="constrained")
for eje, (estrategia, nombre) in zip(ejes, STRATEGIES.items()):
    filas = comparacion.loc[comparacion.strategy.eq(estrategia)]
    colores = ["#204d74" if caso == "central" else "#72a6c9" for caso in filas.scenario]
    eje.barh(filas.scenario_label, 100 * filas.annualized_return, color=colores)
    central = float(filas.loc[filas.scenario.eq("central"), "annualized_return"].iloc[0])
    eje.axvline(100 * central, color="#333333", linestyle="--", label="Central")
    eje.set(title=nombre, xlabel="Rentabilidad anualizada (%)")
    eje.grid(axis="x", alpha=0.25)
ejes[0].invert_yaxis()
ejes[0].legend()
fig.suptitle("Sensibilidad de las reglas sin filtrar · desarrollo 1927–1989")
plt.show()


## 4. Cómo leer los resultados

El préstamo no cambia comprar y mantener ni momentum, porque estas dos reglas no venden en corto. Sí puede cambiar reversión. Excluir dividendos elimina lo que cobra un largo y lo que debe pagar un corto, es decir, una mejora de la reversión en ese caso no demuestra que hayamos descubierto una señal mejor.

Cambiar un supuesto cada vez permite entender su efecto, pero no mide todas sus interacciones. Tampoco corrige la aproximación de Shiller ni demuestra robustez fuera de muestra. No elegiremos después la tarifa más favorable para presentar los resultados.

Esta comparación completa una comprobación de las referencias sin filtrar. Quedan las alternativas sencillas basadas en condiciones de mercado, las variables y etiquetas, los modelos y su validación temporal. La sensibilidad al retraso de señales sigue pendiente.
